In [2]:
"""
retrieval_cartpole.py
---------------------
Minimal retrieval code "good enough" for CartPole:
- Stores a small semantic library of hints and reward templates
- Builds a TF-IDF index over (text + expression fingerprints)
- Retrieves top-k templates given a query (goals/context)
- Suggests a seed expression and SR operator set

Dependencies: numpy, scikit-learn, sympy (optional)

Normalized variable names expected by your CartPole wrapper/notebook:
  x_n, x_dot_n, theta_n, theta_dot_n, u_n
"""

import json
import re
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    import sympy as sp
except Exception:  # sympy is optional for fingerprinting; we fall back to regex
    sp = None

# --------------------------- Data Structures ---------------------------

@dataclass
class Template:
    tid: str
    text: str                        # natural-language hint
    expr_template: str               # symbolic expression (string) using *_n variables
    operators: List[str]             # allowed operators for SR seeding
    tags: List[str]                  # simple keywords for retrieval

@dataclass
class ResultEntry:
    tid: str                         # which template this run used
    context: Dict[str, Any]          # env_id, algo, hparams (optional)
    outcomes: Dict[str, float]       # AUC, final_return, steps_to_thresh, etc.
    expr_final: str                  # final discovered expression (optional)

# --------------------------- Expression Fingerprints ---------------------------

_OP_TOKENS = {
    "+": "op_add", "-": "op_sub", "*": "op_mul", "/": "op_div",
    "sin": "op_sin", "cos": "op_cos", "abs": "op_abs", "tanh": "op_tanh",
    "wrap": "op_wrap", "**2": "op_square"
}
_VAR_TOKENS = ["x_n", "x_dot_n", "theta_n", "theta_dot_n", "u_n"]

def expr_fingerprint(expr_str: str) -> List[str]:
    """
    Tokenize an expression into a small bag of operators & variables.
    Tries SymPy parse; if unavailable or fails, uses regex fallback.
    """
    toks = []
    # Variables
    for v in _VAR_TOKENS:
        if v in expr_str:
            toks.append(f"var_{v}")
    # Operators (cheap scan)
    for k, tk in _OP_TOKENS.items():
        if k == "**2":
            if re.search(r"\*\*2\b", expr_str):
                toks.append(tk)
        else:
            if k in expr_str:
                toks.append(tk)
    # Optional: SymPy parse to detect functions/structure more robustly
    if sp is not None:
        try:
            sy = sp.sympify(expr_str, locals={"wrap": sp.Function('wrap'), "abs": sp.Abs})
            # Collect function names
            for node in sy.atoms(sp.Function):
                name = getattr(node, "name", str(node))
                if name in _OP_TOKENS:
                    toks.append(_OP_TOKENS[name])
        except Exception:
            pass
    return toks

def pack_for_index(t: Template) -> str:
    """
    Combine text, tags, and expression fingerprint into a single string
    for TF-IDF indexing.
    """
    toks = expr_fingerprint(t.expr_template)
    return " ".join([t.text] + t.tags + toks)

# --------------------------- Semantic Library ---------------------------

class SemanticLibrary:
    def __init__(self, store_dir: str = "semantic_lib_cartpole"):
        self.store_dir = Path(store_dir)
        self.store_dir.mkdir(parents=True, exist_ok=True)
        self.templates: Dict[str, Template] = {}
        self.results: List[ResultEntry] = []
        self._vectorizer = None
        self._matrix = None
        self._ids: List[str] = []

    # ---- Persistence ----
    def save(self):
        tmp = {
            "templates": {k: asdict(v) for k, v in self.templates.items()},
            "results": [asdict(r) for r in self.results],
        }
        (self.store_dir / "library.json").write_text(json.dumps(tmp, indent=2))

    def load(self):
        path = self.store_dir / "library.json"
        if not path.exists():
            return
        data = json.loads(path.read_text())
        self.templates = {k: Template(**v) for k, v in data.get("templates", {}).items()}
        self.results = [ResultEntry(**r) for r in data.get("results", [])]

    # ---- Templates & Results ----
    def add_template(self, t: Template, overwrite: bool = False):
        if overwrite or t.tid not in self.templates:
            self.templates[t.tid] = t

    def add_result(self, r: ResultEntry):
        self.results.append(r)

    # ---- Indexing ----
    def build_index(self):
        if not self.templates:
            raise ValueError("No templates to index.")
        docs = []
        ids = []
        for tid, t in self.templates.items():
            docs.append(pack_for_index(t))
            ids.append(tid)
        self._vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1)
        self._matrix = self._vectorizer.fit_transform(docs)
        self._ids = ids
        return self

    # ---- Query ----
    def _query_to_text(self, query: Dict[str, Any]) -> str:
        """
        Convert a structured query into tokens:
        - goals: list[str], e.g., ['upright', 'low_action', 'fast_convergence']
        - context: e.g., env_id='CartPole-v1'
        - constraints: e.g., ['angle_periodic', 'unit_consistent']
        - operator_hints: e.g., ['abs','sin','cos']
        """
        parts = []
        goals = query.get("goals", [])
        parts.extend(goals)
        ctx = query.get("context", {})
        for k, v in ctx.items():
            parts.append(f"{k}:{v}")
        cons = query.get("constraints", [])
        parts.extend(cons)
        ops = query.get("operator_hints", [])
        parts.extend([f"op_{o}" for o in ops])
        return " ".join(parts)

    def retrieve(self, query: Dict[str, Any], topk: int = 3) -> List[Tuple[str, float, Template]]:
        if self._vectorizer is None or self._matrix is None:
            self.build_index()
        qtext = self._query_to_text(query)
        qvec = self._vectorizer.transform([qtext])
        sims = cosine_similarity(qvec, self._matrix)[0]
        order = np.argsort(-sims)[:topk]
        out = []
        for idx in order:
            tid = self._ids[idx]
            out.append((tid, float(sims[idx]), self.templates[tid]))
        return out

    # ---- Synthesis ----
    def suggest_seed(self, query: Dict[str, Any], topk: int = 3) -> Dict[str, Any]:
        """
        Retrieve top-k templates and synthesize:
          - seed_expression: choose best expr or a weighted blend (here: best expr)
          - operator_set: union of operators across top-k
        """
        hits = self.retrieve(query, topk=topk)
        if not hits:
            raise ValueError("No hits found.")
        seed_expr = hits[0][2].expr_template
        op_set = set()
        for _, _, t in hits:
            op_set.update(t.operators)
        return {
            "seed_expression": seed_expr,
            "operators": sorted(op_set),
            "hits": [(tid, score) for tid, score, _ in hits],
        }

# --------------------------- Pre-populated CartPole Templates ---------------------------

def default_cartpole_library() -> SemanticLibrary:
    lib = SemanticLibrary()

    # Upright + small angle velocity
    lib.add_template(Template(
        tid="upright_low_vel",
        text="Keep the pole upright with small angular velocity; minimal control effort.",
        expr_template="- (wrap(theta_n))**2 - 0.1*(theta_dot_n)**2 - 0.01*abs(u_n)",
        operators=["abs", "sin", "cos", "+", "-", "*"],
        tags=["upright", "stability", "low_action", "angle_periodic"]
    ))

    # Center the cart and slow it down
    lib.add_template(Template(
        tid="center_cart",
        text="Keep the cart near the origin and reduce cart velocity.",
        expr_template="- 0.1*(x_n)**2 - 0.05*(x_dot_n)**2",
        operators=["+", "-", "*"],
        tags=["centering", "position_control"]
    ))

    # Combined: upright + center + low action
    lib.add_template(Template(
        tid="upright_center_lowaction",
        text="Upright pole, centered cart, modest action magnitude.",
        expr_template="- (wrap(theta_n))**2 - 0.1*(theta_dot_n)**2 - 0.1*(x_n)**2 - 0.05*(x_dot_n)**2 - 0.01*abs(u_n)",
        operators=["abs", "sin", "cos", "+", "-", "*"],
        tags=["upright", "centering", "low_action", "angle_periodic"]
    ))

    # Smooth actions preference (proxy via |u| and small quadratic velocity terms)
    lib.add_template(Template(
        tid="smooth_actions",
        text="Discourage large actions; prefer smoother control.",
        expr_template="- 0.02*abs(u_n) - 0.02*(x_dot_n**2 + theta_dot_n**2)",
        operators=["abs", "+", "-", "*"],
        tags=["low_action", "smooth", "effort"]
    ))

    # Angle-only template if you want a very sparse expression
    lib.add_template(Template(
        tid="angle_sparse",
        text="Sparse term focusing on angle error only.",
        expr_template="- (wrap(theta_n))**2",
        operators=["sin", "cos", "+", "-", "*"],
        tags=["upright", "sparse", "angle_periodic"]
    ))

    lib.build_index()
    return lib

# --------------------------- Example Usage ---------------------------

if __name__ == "__main__":
    lib = default_cartpole_library()
    query = {
        "goals": ["upright", "centering", "low_action"],
        "context": {"env_id": "CartPole-v1", "algo": "DQN"},
        "constraints": ["angle_periodic"],
        "operator_hints": ["abs", "sin", "cos"]
    }
    suggestion = lib.suggest_seed(query, topk=3)
    print("Suggested seed expression:\n", suggestion["seed_expression"])
    print("Operator set for SR:\n", suggestion["operators"])
    print("Top hits:", suggestion["hits"])


Suggested seed expression:
 - (wrap(theta_n))**2 - 0.1*(theta_dot_n)**2 - 0.1*(x_n)**2 - 0.05*(x_dot_n)**2 - 0.01*abs(u_n)
Operator set for SR:
 ['*', '+', '-', 'abs', 'cos', 'sin']
Top hits: [('upright_center_lowaction', 0.42174193369415014), ('upright_low_vel', 0.18297901935554978), ('smooth_actions', 0.08450554386874054)]
